## SQL aggregates: COUNT, SUM, AVG with read_sql

In [1]:
import sqlite3
import pandas as pd
con = sqlite3.connect("power.db")
sql = """
SELECT COUNT(*) AS n_rows,
ROUND(SUM(gwh)) AS total,
ROUND(AVG(gwh), 1) AS average
FROM generation
WHERE month LIKE '2024%'
"""
print(pd.read_sql(sql, con))

   n_rows      total  average
0    1561  1804424.0   1155.9


## SQL GROUP BY: one total per fuel

In [2]:
sql = """
SELECT fuel, ROUND(SUM(gwh)) AS total
FROM generation
WHERE month LIKE '2024%'
GROUP BY fuel
ORDER BY total DESC
"""
print(pd.read_sql(sql, con))

               fuel      total
0              Coal  1327956.0
1             Hydro   155797.0
2             Solar   133795.0
3              Wind    81544.0
4           Nuclear    54704.0
5               Gas    33929.0
6         Bioenergy    13479.0
7  Other Renewables     2857.0
8      Other Fossil      363.0


## CASE WHEN: coal's share of every year

In [3]:
sql = """
SELECT substr(month, 1, 4) AS year,
ROUND(100 * SUM(CASE
WHEN fuel = 'Coal' THEN gwh
ELSE 0 END)/SUM(gwh), 1) AS coal_pct
FROM generation
GROUP BY year
"""
print(pd.read_sql(sql, con))

   year  coal_pct
0  2019      72.5
1  2020      70.6
2  2021      73.0
3  2022      73.0
4  2023      74.2
5  2024      73.6
6  2025      71.4


## SQL HAVING: keep the big solar states

In [4]:
sql = """
SELECT state, ROUND(SUM(gwh)) AS solar
FROM generation
WHERE fuel = 'Solar'
AND month LIKE '2024%'
GROUP BY state
HAVING SUM(gwh) > 10000
ORDER BY solar DESC
"""
print(pd.read_sql(sql, con))

  state    solar
0    RJ  46040.0
1    GJ  17374.0
2    KA  15398.0
3    TN  14476.0
